# Evaluation
### Imports and config

In [1]:
import torch
import torchaudio
import torchaudio.transforms as T
import numpy as np
import pandas as pd
import os
from pathlib import Path

CHECKPOINT_DIR  = "../models/"
CHECKPOINT      = "best_model.pth"
DEVICE          = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Must match training exactly
SAMPLE_RATE     = 32_000
CHUNK_DURATION  = 5
CHUNK_SAMPLES   = SAMPLE_RATE * CHUNK_DURATION
N_FFT           = 1024
HOP_LENGTH      = 320
N_MELS          = 128
F_MIN           = 50
F_MAX           = 16_000

mel_transform = T.MelSpectrogram(
    sample_rate=SAMPLE_RATE,
    n_fft=N_FFT,
    hop_length=HOP_LENGTH,
    n_mels=N_MELS,
    f_min=F_MIN,
    f_max=F_MAX,
).to(DEVICE)
power_to_db = T.AmplitudeToDB(stype="power", top_db=80).to(DEVICE)

### Load checkpoint and rebuild model

In [2]:
checkpoint = torch.load(os.path.join(CHECKPOINT_DIR, CHECKPOINT), map_location="cpu")
label2idx = checkpoint["label2idx"]
idx2label = {v: k for k, v in label2idx.items()}
NUM_CLASSES = len(label2idx)

print(f"Checkpoint from epoch: {checkpoint['epoch']}")
print(f"Val loss: {checkpoint['val_loss']:.4f}")
print(f"Number of classes: {NUM_CLASSES}")

Checkpoint from epoch: 9
Val loss: 0.0046
Number of classes: 205


In [3]:
import timm

model = timm.create_model(
    "efficientnet_b0",
    pretrained=False, 
    num_classes=NUM_CLASSES, 
    in_chans=3
)
model.load_state_dict(checkpoint["model_state"])
model.to(DEVICE)
model.eval()
print("Model loaded successfully")

Model loaded successfully


### Inference

In [4]:
def predict(audio_path, start_sample=0, top_k=5):
    waveform, sr = torchaudio.load(audio_path)
    if sr != SAMPLE_RATE:
        waveform = torchaudio.functional.resample(waveform, sr, SAMPLE_RATE)
    
    if waveform.shape[0] > 1:
        waveform = waveform.mean(dim=0, keepdim=True)
    
    # Extract the specific chunk using start_sample
    end_sample = start_sample + CHUNK_SAMPLES
    if end_sample <= waveform.shape[1]:
        waveform = waveform[:, start_sample:end_sample]
    else:
        waveform = waveform[:, start_sample:]
        pad_size = CHUNK_SAMPLES - waveform.shape[1]
        waveform = torch.nn.functional.pad(waveform, (0, pad_size))
    
    waveform = waveform.to(DEVICE)
    mel = mel_transform(waveform)
    mel_db = power_to_db(mel)
    mel_db = (mel_db - mel_db.mean()) / (mel_db.std() + 1e-6)
    mel_db = mel_db.repeat(3, 1, 1)
    tensor = mel_db.unsqueeze(0).to(DEVICE)
    
    with torch.no_grad():
        logits = model(tensor)
        probs = torch.softmax(logits, dim=1).squeeze().cpu().numpy()
    
    top_indices = probs.argsort()[-top_k:][::-1]
    results = [(idx2label[i], float(probs[i])) for i in top_indices]
    return results

In [5]:
from torch.utils.data import Dataset, DataLoader

class EvalDataset(Dataset):
    def __init__(self, df, audio_dir):
        self.df = df.reset_index(drop=True)
        self.audio_dir = audio_dir

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        filepath = os.path.join(self.audio_dir, row['filename'].strip())
        label = row['primary_label'].strip()
        start_sample = int(row['start_sample'])

        waveform, _ = librosa.load(
            filepath,
            sr=SAMPLE_RATE,
            mono=True,
            offset=start_sample / SAMPLE_RATE,
            duration=CHUNK_DURATION,
        )
        if len(waveform) < CHUNK_SAMPLES:
            repeats = (CHUNK_SAMPLES // len(waveform)) + 1
            waveform = np.tile(waveform, repeats)[:CHUNK_SAMPLES]

        waveform = torch.from_numpy(waveform.astype(np.float32)).unsqueeze(0)
        return waveform, label

### Test on a sample

In [6]:
# Pick any audio file from your dataset
test_file = "../dataset/birdclef-2025/train_audio/saffin/XC1768.ogg"

results = predict(test_file)
for species, prob in results:
    print(f"{species}: {prob:.4f}")

babwar: 0.6820
yebsee1: 0.0639
verfly: 0.0622
135045: 0.0471
purgal2: 0.0175


### Evaluate on test_manifest.csv

In [7]:
import librosa

AUDIO_DIR = "../dataset/birdclef-2025/train_audio"
EVAL_BATCH_SIZE = 64

test_df = pd.read_csv("../dataset/test_manifest.csv")
eval_dataset = EvalDataset(test_df, AUDIO_DIR)
eval_loader = DataLoader(eval_dataset, batch_size=EVAL_BATCH_SIZE, num_workers=8, pin_memory=True)

model.eval()
top_k_correct = {1: 0, 3: 0, 5: 0}
total = 0

with torch.no_grad():
    for batch_waveforms, batch_labels in eval_loader:
        batch_waveforms = batch_waveforms.to(DEVICE)
        mel = mel_transform(batch_waveforms)
        mel_db = power_to_db(mel)
        mel_db = (mel_db - mel_db.min()) / (mel_db.max() - mel_db.min() + 1e-6)
        mel_db = mel_db.repeat(1, 3, 1, 1)

        logits = model(mel_db)
        probs = torch.sigmoid(logits)

        for i, true_label in enumerate(batch_labels):
            true_idx = label2idx[true_label.strip()]
            sorted_indices = probs[i].argsort(descending=True)
            for k in [1, 3, 5]:
                if true_idx in sorted_indices[:k]:
                    top_k_correct[k] += 1
            total += 1

for k in [1, 3, 5]:
    print(f"Top-{k} accuracy: {top_k_correct[k]/total:.4f}")

Top-1 accuracy: 0.8784
Top-3 accuracy: 0.9347
Top-5 accuracy: 0.9506


In [10]:
# Look at a few individual predictions
model.eval()
sample_count = 0

with torch.no_grad():
    for batch_waveforms, batch_labels in eval_loader:
        batch_waveforms = batch_waveforms.to(DEVICE)
        mel = mel_transform(batch_waveforms)
        mel_db = power_to_db(mel)
        mel_db = (mel_db - mel_db.mean()) / (mel_db.std() + 1e-6)
        mel_db = mel_db.repeat(1, 3, 1, 1)

        logits = model(mel_db)
        probs = torch.sigmoid(logits)

        for i, true_label in enumerate(batch_labels):
            true_idx = label2idx[true_label.strip()]
            top5_indices = probs[i].argsort(descending=True)[:5]
            top5 = [(idx2label[idx.item()], probs[i, idx].item()) for idx in top5_indices]
            true_prob = probs[i, true_idx].item()
            
            print(f"True: {true_label.strip()} (prob: {true_prob:.4f})")
            print(f"Top5: {top5}")
            print()
            
            sample_count += 1
            if sample_count >= 10:
                break
        if sample_count >= 10:
            break

True: stbwoo2 (prob: 0.0128)
Top5: [('babwar', 0.9799930453300476), ('555086', 0.7467021942138672), ('speowl1', 0.6238894462585449), ('1346504', 0.5376729369163513), ('41663', 0.42233070731163025)]

True: tbsfin1 (prob: 0.0003)
Top5: [('babwar', 0.982394278049469), ('turvul', 0.4142336845397949), ('42007', 0.33231741189956665), ('whtdov', 0.23459556698799133), ('rufmot1', 0.06831241399049759)]

True: chbant1 (prob: 0.0000)
Top5: [('whtdov', 7.756467151303164e-22), ('roahaw', 1.2472579840889215e-37), ('1139490', 0.0), ('1192948', 0.0), ('1194042', 0.0)]

True: saffin (prob: 0.0632)
Top5: [('babwar', 0.8267927765846252), ('555086', 0.807032585144043), ('67082', 0.7183489799499512), ('secfly1', 0.6116752028465271), ('eardov1', 0.6015526652336121)]

True: strfly1 (prob: 0.0000)
Top5: [('blbgra1', 0.36182060837745667), ('24322', 0.006393571384251118), ('greegr', 0.00398427527397871), ('1192948', 0.0036708551924675703), ('blkvul', 0.0019620275124907494)]

True: crbtan1 (prob: 0.0000)
Top5: [

In [11]:
train_df = pd.read_csv("../dataset/train_manifest.csv")
print(train_df['primary_label'].value_counts().head(10))
print(train_df['primary_label'].value_counts().tail(10))

primary_label
compau     4879
grekis     4371
roahaw     4210
whtdov     3990
laufal1    3882
trokin     3673
yercac1    3204
trsowl     3093
banana     3015
wbwwre1    2853
Name: count, dtype: int64
primary_label
65419     7
67082     7
81930     7
42113     3
42087     3
523060    3
548639    3
868458    3
66016     3
21116     1
Name: count, dtype: int64


In [12]:
print(train_df['primary_label'].value_counts()['babwar'])
print(f"Rank: {list(train_df['primary_label'].value_counts().index).index('babwar') + 1}")

966
Rank: 52


In [13]:
print(f"babwar index: {label2idx['babwar']}")
print(f"Index 0 label: {idx2label[0]}")

babwar index: 63
Index 0 label: 1139490


In [19]:
waveform, sr = torchaudio.load("../dataset/birdclef-2025/train_audio/saffin/XC1768.ogg")
waveform = waveform.to(DEVICE)
mel = mel_transform(waveform)
mel_db = power_to_db(mel)

# Training normalization (min-max)
mel_normalized = (mel_db - mel_db.min()) / (mel_db.max() - mel_db.min() + 1e-6)
mel_normalized = mel_normalized.repeat(3, 1, 1).unsqueeze(0)

with torch.no_grad():
    logits = model(mel_normalized)
    probs = torch.sigmoid(logits)
    top5 = probs[0].argsort(descending=True)[:5]
    for idx in top5:
        print(f"{idx2label[idx.item()]}: {probs[0, idx].item():.4f}")

saffin: 0.9967
baymac: 0.0001
paltan1: 0.0000
grepot1: 0.0000
rinkin1: 0.0000
